# Stany magazynowe

Krok ETL po `transform`. Dodaje do tabeli dwie kolumny stanu magazynowego:
- **`StanPoRuchu`** — stan towaru po każdym wierszu, narastająco w kolejności `Data` → `KolejnyWDniu` → `Kolejnosc`,
- **`StanKoniecDnia`** — ostatni `StanPoRuchu` dnia, powtórzony we wszystkich wierszach SKU × dzień (w analizach **nie sumować**, brać `first()` / `max()`),
- **`StanPoczatekDnia`** — stan przed pierwszym ruchem dnia (= `StanKoniecDnia` poprzedniego dnia); bez sprzedaży z danego dnia, więc bez wycieku jako cecha do prognozy,
- **`KorektaRemanentu`** — o ile REM/BO zmienił stan wyliczony z ruchów (≠ 0 tylko w dniu resetu); ujemna = braki, dodatnia = nadwyżki,
- **`DniOdResetu`** — dni od ostatniego REM/BO towaru (puste przed pierwszym resetem); im więcej, tym mniej wiarygodny stan.

Kolumny dnia (`StanKoniecDnia`, `StanPoczatekDnia`, `KorektaRemanentu`, `DniOdResetu`) są powtórzone we wszystkich wierszach SKU × dzień — **nie sumować**. Zawsze: `StanPoczatekDnia` + ruchy dnia + `KorektaRemanentu` = `StanKoniecDnia`.

**Metoda:**
- ruch = `RuchMagazynowy` (z `transform`, ze znakiem); dokumenty nieaktywne (`AktywnyDok = 0`) nie zmieniają stanu,
- REM i BO (`MetodaLiczenia = 'RESET_IP'`) **resetują stan** do ilości ze spisu — w wierszu resetu `RuchMagazynowy` = `IloscPlus` (stan ze spisu, nie zmiana); pozycje tego samego towaru w jednym dokumencie się sumują,
- pusta pozycja REM (`IloscPlus = 0` i `IloscMinus = 0`) też resetuje — spis 0 oznacza stan 0; puste pozycje BO pomijamy (`BO/23/2` to pusty bilans otwarcia magazynu 2, `MagId = 2`, który wyzerowałby stan startowy z `BO/23/1`),
- wiersze zerowe z `calendar` (`DokId = -1`, `KolejnyWDniu = -1`) mają ruch 0 — przenoszą stan z poprzedniego dnia,
- przed pierwszym resetem towaru stan liczony od 0.

**Kontrola jakości:** stan wyliczony tuż przed każdym REM vs `IloscMinus` (stan systemowy przed spisem).

Wejście: `dane/interim/fact_inka_final.parquet`. Wyjście: `dane/interim/fact_inka_final_stany.parquet` (wejście + 2 kolumny, ta sama liczba wierszy). Szczegóły i wyniki: `docs/stany_hiu.md`.

In [1]:
# Importy i ustawienie katalogu głównego projektu
import sys, os
import pandas as pd
import numpy as np

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools

## [PARAMETRY]

In [2]:
WEJSCIE      = "dane/interim/fact_inka_final.parquet"
WYJSCIE      = "dane/interim/fact_inka_final_stany.parquet"
KOLUMNY_HASH = ['SKU', 'Data', 'DokId', 'Kolejnosc']    # sortowanie do sumy kontrolnej (jak w transform)
KOLEJNOSC    = ['SKU', 'Data', 'KolejnyWDniu', 'Kolejnosc']   # kolejność ruchów przy liczeniu stanu
ZAOKRAGLENIE = 4        # miejsca po przecinku — sumy ułamków (towary ważone) dają np. 2.34499999
TOLERANCJA   = 0.001    # różnica uznawana za zgodność w kontroli REM
TYPDOK_REM   = 16       # remanent
TYPDOK_BO    = 14       # bilans otwarcia — też resetuje, ale nie ma stanu systemowego do porównania

## [FUNKCJE]

In [3]:
def ruch_do_stanu(df):
    """Zmiana stanu w wierszu: RuchMagazynowy dla aktywnych dokumentów, 0 dla pozostałych.

    W wierszach resetu (RESET_IP) wartość = ilość ze spisu — sumowana od zera w nowej epoce.
    """
    return df['RuchMagazynowy'].where(df['AktywnyDok'] == 1, 0.0)


def czy_reset(df):
    """Wiersze resetujące stan: pozycja REM/BO z aktywnego dokumentu.

    Pusta pozycja REM (IloscPlus = 0 i IloscMinus = 0) też resetuje — spis 0 oznacza stan 0.
    Puste pozycje BO pomijamy — BO/23/2 to pusty bilans otwarcia magazynu 2 (MagId = 2),
    który wyzerowałby stan startowy z BO/23/1 (magazyn 1).
    """
    pusta = (df['IloscPlus'] == 0) & (df['IloscMinus'] == 0)
    pusty_bo = pusta & (df['TypDok'] == TYPDOK_BO)
    return (df['MetodaLiczenia'] == 'RESET_IP') & (df['AktywnyDok'] == 1) & ~pusty_bo


def poczatek_resetu(df, reset):
    """Pierwsza pozycja resetu w dokumencie (SKU × DokId) — od niej zaczyna się nowa epoka.

    Kolejne pozycje tego samego dokumentu dodają się do stanu (spis sumowany po pozycjach).
    Wymaga posortowanego df.
    """
    pierwsza_resetu = ~df[reset].duplicated(['SKU', 'DokId'])
    return reset & pierwsza_resetu.reindex(df.index, fill_value=False)


def stan_po_ruchu(df, ruch, start):
    """Stan po każdym wierszu: suma narastająca ruchów w epoce (od resetu do resetu) per SKU.

    Przed pierwszym resetem stan liczony od 0. Wymaga posortowanego df.
    """
    epoka = start.groupby(df['SKU']).cumsum()
    return ruch.groupby([df['SKU'], epoka]).cumsum().round(ZAOKRAGLENIE)


def stan_koniec_dnia(df):
    """Ostatni StanPoRuchu dnia, powtórzony we wszystkich wierszach SKU × Data. Wymaga posortowanego df."""
    return df.groupby(['SKU', 'Data'])['StanPoRuchu'].transform('last')


def stan_poczatek_dnia(df):
    """StanKoniecDnia poprzedniego dnia towaru (pierwszy dzień: 0), powtórzony we wszystkich wierszach dnia.
    Wymaga posortowanego df."""
    koniec_dnia = df.drop_duplicates(['SKU', 'Data'], keep='last')[['SKU', 'Data', 'StanKoniecDnia']]
    poczatek = koniec_dnia.groupby('SKU')['StanKoniecDnia'].shift(1).fillna(0.0)
    mapa = pd.Series(poczatek.to_numpy(), index=pd.MultiIndex.from_frame(koniec_dnia[['SKU', 'Data']]))
    return pd.Series(mapa.reindex(pd.MultiIndex.from_frame(df[['SKU', 'Data']])).to_numpy(), index=df.index)


def korekta_remanentu(df, ruch, reset):
    """Zmiana stanu wynikająca z REM/BO w danym dniu: koniec dnia − początek dnia − ruchy dnia (bez wierszy resetu).
    Różna od 0 tylko w dniach z resetem."""
    ruchy_dnia = ruch.where(~reset, 0.0).groupby([df['SKU'], df['Data']]).transform('sum')
    return (df['StanKoniecDnia'] - df['StanPoczatekDnia'] - ruchy_dnia).round(ZAOKRAGLENIE)


def dni_od_resetu(df, reset):
    """Dni od ostatniego REM/BO towaru (do danego dnia włącznie); przed pierwszym resetem brak wartości."""
    data_resetu = df['Data'].where(reset).groupby(df['SKU']).ffill()                  # data ostatniego resetu do tego wiersza
    data_resetu = data_resetu.groupby([df['SKU'], df['Data']]).transform('max')   # reset w ciągu dnia liczy się od rana
    return (df['Data'] - data_resetu).dt.days.astype('Int16')


def kontrola_rem(df, reset, start):
    """Stan wyliczony tuż przed każdym REM vs stan systemowy przed spisem (suma IloscMinus dokumentu).

    Zwraca jeden wiersz na SKU × dokument REM z kolumnami StanWyliczony, StanSystemowy, Roznica, Zgodny.
    """
    stan_przed = df.groupby('SKU')['StanPoRuchu'].shift(1).fillna(0.0)   # stan po poprzednim wierszu
    rem = df[reset & (df['TypDok'] == TYPDOK_REM)]
    wynik = rem.groupby(['SKU', 'DokId'], as_index=False).agg(
        Data=('Data', 'first'), CzyWazony=('CzyWazony', 'first'), StanSystemowy=('IloscMinus', 'sum'))
    pierwsza_rem = start & (df['TypDok'] == TYPDOK_REM)
    przed = df.loc[pierwsza_rem, ['SKU', 'DokId']].assign(StanWyliczony=stan_przed[pierwsza_rem])
    wynik = wynik.merge(przed, on=['SKU', 'DokId'], how='left')
    wynik['Roznica'] = (wynik['StanWyliczony'] - wynik['StanSystemowy']).round(ZAOKRAGLENIE)
    wynik['Zgodny'] = wynik['Roznica'].abs() <= TOLERANCJA
    return wynik

## Wczytanie

In [4]:
# Suma kontrolna wejścia — ten sam hash u obu osób = identyczne dane
moj_hash = tools.hash_danych_bezpieczny(WEJSCIE, kolumny_sortowania=KOLUMNY_HASH)
print(f"Mój hash (posortowane):   {os.path.basename(WEJSCIE)}   {moj_hash}")

Mój hash (posortowane):   fact_inka_final.parquet   b73691d7bc6fef407c5e40bb256ad261b4c1c28537d0a19b98ef5ec89bf62bef


Hash `fact_inka_final.parquet` (z wyjścia `transform.ipynb`):  
**b73691d7bc6fef407c5e40bb256ad261b4c1c28537d0a19b98ef5ec89bf62bef**

In [5]:
df = pd.read_parquet(WEJSCIE)
n_wejscie = len(df)
print(f"Wierszy: {n_wejscie:,}, kolumn: {df.shape[1]}")

Wierszy: 9,485,872, kolumn: 60


## Obliczenia

Sortowanie `SKU` → `Data` → `KolejnyWDniu` → `Kolejnosc` (wiersz zerowy dnia ma `KolejnyWDniu = -1`, więc stoi na początku dnia).

In [6]:
df = df.sort_values(KOLEJNOSC, kind='stable').reset_index(drop=True)

ruch  = ruch_do_stanu(df)
reset = czy_reset(df)
start = poczatek_resetu(df, reset)

df['StanPoRuchu']    = stan_po_ruchu(df, ruch, start)
df['StanKoniecDnia'] = stan_koniec_dnia(df)
df['StanPoczatekDnia'] = stan_poczatek_dnia(df)
df['KorektaRemanentu'] = korekta_remanentu(df, ruch, reset)
df['DniOdResetu']      = dni_od_resetu(df, reset)

print(f"Pozycji resetu: {reset.sum():,}, dokumentów resetu (SKU × DokId): {start.sum():,}")

Pozycji resetu: 33,127, dokumentów resetu (SKU × DokId): 33,126


## [WALIDACJA]

In [7]:
# Kontrole techniczne
assert len(df) == n_wejscie, "Liczba wierszy różna od wejścia"
assert df['StanPoRuchu'].notna().all() and df['StanKoniecDnia'].notna().all(), "NaN w stanach"
assert (df.groupby(['SKU', 'Data'])['StanKoniecDnia'].nunique() == 1).all(), "StanKoniecDnia niestały w dniu"

# w pierwszej pozycji resetu stan = ilość ze spisu
assert np.allclose(df.loc[start, 'StanPoRuchu'], df.loc[start, 'RuchMagazynowy'].round(ZAOKRAGLENIE)), "Reset nie ustawia stanu"

# StanKoniecDnia = StanPoRuchu w ostatnim wierszu dnia
ostatni_w_dniu = ~df.duplicated(['SKU', 'Data'], keep='last')
assert (df.loc[ostatni_w_dniu, 'StanKoniecDnia'] == df.loc[ostatni_w_dniu, 'StanPoRuchu']).all(), "StanKoniecDnia ≠ ostatni StanPoRuchu"
# StanPoczatekDnia = StanKoniecDnia poprzedniego dnia towaru (pierwszy dzień: 0)
koniec_dnia = df.drop_duplicates(['SKU', 'Data'], keep='last')
assert np.allclose(koniec_dnia['StanPoczatekDnia'], koniec_dnia.groupby('SKU')['StanKoniecDnia'].shift(1).fillna(0.0)), "StanPoczatekDnia ≠ koniec poprzedniego dnia"

# korekta różna od 0 tylko w dniach z resetem
dzien_resetu = reset.groupby([df['SKU'], df['Data']]).transform('any')
assert (df.loc[~dzien_resetu, 'KorektaRemanentu'].abs() <= TOLERANCJA).all(), "KorektaRemanentu poza dniem resetu"

# DniOdResetu: 0 w dniu resetu, brak wartości przed pierwszym resetem
assert (df.loc[dzien_resetu, 'DniOdResetu'] == 0).all(), "DniOdResetu ≠ 0 w dniu resetu"
print("Kontrole techniczne OK")

Kontrole techniczne OK


In [8]:
# Kontrola jakości: stan wyliczony przed REM vs stan systemowy (IloscMinus)
kontrola = kontrola_rem(df, reset, start)
print(f"REM (SKU × dokument): {len(kontrola):,}, zgodnych: {kontrola['Zgodny'].mean():.1%}")

zgodnosc_wg_wagi = (kontrola.groupby('CzyWazony')['Zgodny']
                    .agg(REM='size', Zgodnych='mean')
                    .rename(index={0: 'nieważone', 1: 'ważone'}))
zgodnosc_wg_wagi['Zgodnych'] = (zgodnosc_wg_wagi['Zgodnych'] * 100).round(1).astype(str) + '%'
print(zgodnosc_wg_wagi)

REM (SKU × dokument): 27,708, zgodnych: 96.2%
             REM Zgodnych
CzyWazony                
nieważone  26918    97.1%
ważone       790    65.9%


In [9]:
# Przegląd: ujemne stany — głównie artefakty danych (dokumenty wpisane później z wcześniejszą datą,
# PZ zbiorczy, start od 0 przed pierwszym REM); szczegóły w docs/stany_hiu.md
dzien = df.drop_duplicates(['SKU', 'Data'])
print(f"SKU-dni z ujemnym StanKoniecDnia: {(dzien['StanKoniecDnia'] < 0).mean():.2%}")
print(f"SKU z ujemnym stanem choć raz: {dzien.loc[dzien['StanKoniecDnia'] < 0, 'SKU'].nunique():,} z {dzien['SKU'].nunique():,}")

SKU-dni z ujemnym StanKoniecDnia: 1.64%
SKU z ujemnym stanem choć raz: 2,850 z 12,481


## Zapis + hash

In [10]:
df.to_parquet(WYJSCIE, compression='zstd', index=False)
print(f"Zapisano: {df.shape}")

Zapisano: (9485872, 65)


In [ ]:
# Suma kontrolna wyniku
moj_hash = tools.hash_danych_bezpieczny(WYJSCIE, kolumny_sortowania=KOLUMNY_HASH)
print(f"Mój hash (posortowane):   {os.path.basename(WYJSCIE)}   {moj_hash}")

Mój hash (posortowane):   fact_inka_final_stany.parquet   d3e0bcc1ed1d93e5011c9e93c563197ce507610b5006a8675fe0db094762779d


Hash końcowy `fact_inka_final_stany.parquet`:  
**d3e0bcc1ed1d93e5011c9e93c563197ce507610b5006a8675fe0db094762779d**